In [ ]:
!pip install -q kaggle

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving kaggle.json to kaggle.json


In [ ]:
import os
import shutil

os.makedirs("/root/.kaggle", exist_ok=True)

shutil.copy(
    "/content/kaggle.json",
    "/root/.kaggle/kaggle.json"
)

os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("Kaggle API is ready")

Kaggle API is ready


In [ ]:
!kaggle datasets download \
    -d saifthesaif/comiset-lab-50-benign-subset \
    -p /content/comiset_dataset

Dataset URL: https://www.kaggle.com/datasets/saifthesaif/comiset-lab-50-benign-subset
License(s): CC0-1.0
100% 1.66G/1.66G [00:25<00:00, 70.8MB/s]



In [ ]:
!unzip -q /content/comiset_dataset/comiset-lab-50-benign-subset.zip \
    -d /content/comiset_dataset

In [ ]:
import glob

PARQUET_DIR = "/content/comiset_dataset/comiset_parquet_v12"

FILES = sorted(
    glob.glob(f"{PARQUET_DIR}/*.parquet")
)

print("Number of input files:", len(FILES))

Number of input files: 130


In [ ]:
import glob
import re
import math
import os

import polars as pl
import pandas as pd

In [ ]:
PARQUET_DIR = "/content/comiset_dataset/comiset_parquet_v12"

FILES = sorted(glob.glob(f"{PARQUET_DIR}/*.parquet"))

print(f"Number of files: {len(FILES)}")

Number of files: 130


##the profiler

In [ ]:
ROWS_PER_FILE = 2000
N_EXAMPLES = 5

print(f"Files: {len(FILES)}, rows/file: {ROWS_PER_FILE}")

INT_RE = re.compile(r"^[+-]?\d+$")
FLOAT_RE = re.compile(r"^[+-]?(?:\d+\.\d*|\.\d+|\d+\.?\d*[eE][+-]?\d+)$")
DATE_RE = re.compile(r"^\d{4}-\d{2}-\d{2}")
BOOLS = {"true", "false"}

def is_int(x):
    return bool(INT_RE.match(x))

def is_float(x):
    return bool(FLOAT_RE.match(x))

def is_date(x):
    return bool(DATE_RE.match(x))

def is_bool(x):
    return x.lower() in BOOLS

def is_identifier(name, values):
    name = name.lower()
    keywords = [
        "id", "guid", "uuid", "hash", "pid", "tid", "uid",
        "identifier", "session", "token"
    ]
    return any(k in name for k in keywords) or (
        values and len(set(values)) / len(values) > 0.8
    )

samples = []

for i, f in enumerate(FILES):
    try:
        samples.append(
            pl.read_parquet(
                f,
                n_rows=ROWS_PER_FILE
            )
        )
    except Exception as e:
        print(f"Failed: {f}\n{e}")

    if (i + 1) % 50 == 0:
        print(f"Read {i + 1}/{len(FILES)} files")

df = pl.concat(samples, how="diagonal_relaxed")

print("Sample shape:", df.shape)

results = []

for i, col in enumerate(df.columns):

    values = [
        str(x)
        for x in df[col].drop_nulls().to_list()
    ]

    n = len(values)
    total = df.height

    if not n:
        results.append({
            "field": col,
            "sample_rows": total,
            "non_null": 0,
            "null_pct": 100,
            "unique_count": 0,
            "unique_ratio": 0,
            "integer_pct": 0,
            "float_pct": 0,
            "numeric_pct": 0,
            "datetime_pct": 0,
            "boolean_pct": 0,
            "identifier_candidate": False,
            "min_numeric": None,
            "max_numeric": None,
            "avg_string_length": None,
            "examples": []
        })
        continue

    ints = sum(is_int(x) for x in values)
    floats = sum(is_float(x) for x in values)

    nums = []

    for x in values:
        try:
            nums.append(float(x))
        except:
            pass

    date_candidates = [
        x for x in values
        if is_date(x)
    ]

    dates = 0

    for x in date_candidates:
        try:
            pd.to_datetime(
                x,
                errors="raise",
                utc=True
            )
            dates += 1
        except:
            pass

    bools = sum(
        is_bool(x)
        for x in values
    )

    unique = set(values)

    results.append({
        "field": col,
        "sample_rows": total,
        "non_null": n,
        "null_pct": 100 * (total - n) / total,
        "unique_count": len(unique),
        "unique_ratio": len(unique) / n,
        "integer_pct": 100 * ints / n,
        "float_pct": 100 * floats / n,
        "numeric_pct": 100 * len(nums) / n,
        "datetime_pct": 100 * dates / n,
        "boolean_pct": 100 * bools / n,
        "identifier_candidate": is_identifier(col, values),
        "min_numeric": min(nums) if nums else None,
        "max_numeric": max(nums) if nums else None,
        "avg_string_length": sum(map(len, values)) / n,
        "examples": list(unique)[:N_EXAMPLES]
    })

    if (i + 1) % 100 == 0:
        print(f"Profiled {i + 1}/{len(df.columns)} columns")

profile = pd.DataFrame(results)

profile.to_parquet(
    "/content/comiset_column_profile.parquet",
    index=False
)

profile.to_csv(
    "/content/comiset_column_profile.csv",
    index=False
)

print("Profile shape:", profile.shape)

display(profile)

import gc

del df
del samples

gc.collect()

Files: 130, rows/file: 2000
Read 50/130 files
Read 100/130 files
Sample shape: (260000, 471)
Profiled 100/471 columns
Profiled 200/471 columns
Profiled 300/471 columns
Profiled 400/471 columns
Profile shape: (471, 16)


,field,sample_rows,non_null,null_pct,unique_count,unique_ratio,integer_pct,float_pct,numeric_pct,datetime_pct,boolean_pct,identifier_candidate,min_numeric,max_numeric,avg_string_length,examples
0,@timestamp,260000,260000,0.000000,163825,0.630096,0.0,0.0,0.0,100.0,0.0,False,NaN,NaN,31.990146,"[2022-11-17 16:08:07.457000+00:00, 2022-11-21 ..."
1,AccountName,260000,30,99.988462,2,0.066667,0.0,0.0,0.0,0.0,0.0,False,NaN,NaN,11.266667,"[localsystem, device owners]"
2,Action,260000,4,99.998462,1,0.250000,100.0,0.0,100.0,0.0,0.0,False,2.0,2.0,1.000000,[2]
3,Action ID,260000,1,99.999615,1,1.000000,100.0,0.0,100.0,0.0,0.0,True,2.0,2.0,1.000000,[2]
4,Action Name,260000,1,99.999615,1,1.000000,0.0,0.0,0.0,0.0,0.0,True,NaN,NaN,5.000000,[%%809]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
466,user_name,260000,202519,22.108077,9,0.000044,0.0,0.0,0.0,0.0,0.0,False,NaN,NaN,6.300298,"[louis, network service, desktop-4pvps6e$, sys..."
467,user_sid,260000,1728,99.335385,2,0.001157,0.0,0.0,0.0,0.0,0.0,True,NaN,NaN,8.043981,[S-1-5-21-1311114680-1922563338-3976463898-100...
468,version,260000,254601,2.076538,5,0.000020,100.0,0.0,100.0,0.0,0.0,False,1.0,5.0,1.000000,"[3, 5, 1, 2, 4]"
469,xml_name,260000,155,99.940385,9,0.058065,0.0,0.0,0.0,0.0,0.0,False,NaN,NaN,19.380645,"[UMDFHostProblem, Operation_ClientFailure, Ope..."


9

###Drop columns

In [ ]:
DROP_COLUMNS = [
    "InstanceId", "SourceThreadId", "TaskInstanceId", "activity_id",
    "hash_imphash", "hash_md5", "hash_sha1", "hash_sha256",
    "process_guid", "process_id", "process_parent_guid", "process_parent_id",
    "target_process_guid", "target_process_id", "thread_id", "updateGuid",
    "EnginePID", "FilterID", "Id", "LogonGuid", "LogonId", "NewThreadId",
    "ProcessID", "Thread ID", "etl_host_agent_ephemeral_uid", "transferId"
]

In [ ]:
import os
import gc
import polars as pl

INPUT_DIR = "/content/comiset_dataset/content/comiset_parquet_v10"
OUTPUT_DIR = "/content/comiset_parquet_v11"

os.makedirs(OUTPUT_DIR, exist_ok=True)

FILES = sorted(
    [os.path.join(INPUT_DIR, f)
     for f in os.listdir(INPUT_DIR)
     if f.endswith(".parquet")]
)

for i, f in enumerate(FILES, start=1):

    df = pl.read_parquet(f)


    cols_to_drop = [
        c for c in DROP_COLUMNS
        if c in df.columns
    ]

    df = df.drop(cols_to_drop)

    output_file = os.path.join(
        OUTPUT_DIR,
        os.path.basename(f)
    )

    df.write_parquet(
        output_file,
        compression="zstd"
    )

    print(
        f"{i}/{len(FILES)} | "
        f"{os.path.basename(f)} | "
        f"shape: {df.shape}"
    )

    del df
    gc.collect()

print("\nDone!")
print("Output:", OUTPUT_DIR)

1/130 | part-00000.parquet | shape: (200000, 445)
2/130 | part-00001.parquet | shape: (200000, 445)
3/130 | part-00002.parquet | shape: (200000, 445)
4/130 | part-00003.parquet | shape: (200000, 445)
5/130 | part-00004.parquet | shape: (200000, 445)
6/130 | part-00005.parquet | shape: (200000, 445)
7/130 | part-00006.parquet | shape: (200000, 445)
8/130 | part-00007.parquet | shape: (200000, 445)
9/130 | part-00008.parquet | shape: (200000, 445)
10/130 | part-00009.parquet | shape: (200000, 445)
11/130 | part-00010.parquet | shape: (200000, 445)
12/130 | part-00011.parquet | shape: (200000, 445)
13/130 | part-00012.parquet | shape: (200000, 445)
14/130 | part-00013.parquet | shape: (200000, 445)
15/130 | part-00014.parquet | shape: (200000, 445)
16/130 | part-00015.parquet | shape: (200000, 445)
17/130 | part-00016.parquet | shape: (200000, 445)
18/130 | part-00017.parquet | shape: (200000, 445)
19/130 | part-00018.parquet | shape: (200000, 445)
20/130 | part-00019.parquet | shape: (20

In [ ]:
import os

OUTPUT_DIR = "/content/comiset_parquet_v11"

output_files = sorted(
    [
        f for f in os.listdir(OUTPUT_DIR)
        if f.endswith(".parquet")
    ]
)

print("Files already saved:", len(output_files))

print("\nLast 5 saved files:")
for f in output_files[-5:]:
    print(f)

Files already saved: 52

Last 5 saved files:
part-00047.parquet
part-00048.parquet
part-00049.parquet
part-00050.parquet
part-00051.parquet


In [ ]:
import os
import gc
import polars as pl

INPUT_DIR = "/content/comiset_dataset/content/comiset_parquet_v10"
OUTPUT_DIR = "/content/comiset_parquet_v11"

os.makedirs(OUTPUT_DIR, exist_ok=True)

FILES = sorted([
    os.path.join(INPUT_DIR, f)
    for f in os.listdir(INPUT_DIR)
    if f.endswith(".parquet")
])

saved_files = set(
    f for f in os.listdir(OUTPUT_DIR)
    if f.endswith(".parquet")
)

remaining_files = [
    f for f in FILES
    if os.path.basename(f) not in saved_files
]

print("Total input files:", len(FILES))
print("Already saved:", len(saved_files))
print("Remaining:", len(remaining_files))

print("\nFirst remaining file:")
print(os.path.basename(remaining_files[0]))

Total input files: 130
Already saved: 52
Remaining: 78

First remaining file:
part-00052.parquet


In [ ]:
import os
import gc
import polars as pl

for i, f in enumerate(remaining_files, start=53):

    output_file = os.path.join(
        OUTPUT_DIR,
        os.path.basename(f)
    )


    if os.path.exists(output_file):
        print(f"{i}/130 | Already exists | {os.path.basename(f)}")
        continue

    try:

        df = pl.read_parquet(f)


        cols_to_drop = [
            c for c in DROP_COLUMNS
            if c in df.columns
        ]

        df = df.drop(cols_to_drop)


        df.write_parquet(
            output_file,
            compression="zstd"
        )

        print(
            f"{i}/130 | "
            f"{os.path.basename(f)} | "
            f"shape: {df.shape}"
        )

    finally:

        if "df" in locals():
            del df

        gc.collect()

print("\nFinished remaining files!")

53/130 | part-00052.parquet | shape: (200000, 445)
54/130 | part-00053.parquet | shape: (200000, 445)
55/130 | part-00054.parquet | shape: (200000, 445)
56/130 | part-00055.parquet | shape: (200000, 445)
57/130 | part-00056.parquet | shape: (200000, 445)
58/130 | part-00057.parquet | shape: (200000, 445)
59/130 | part-00058.parquet | shape: (200000, 445)
60/130 | part-00059.parquet | shape: (200000, 445)
61/130 | part-00060.parquet | shape: (200000, 445)
62/130 | part-00061.parquet | shape: (200000, 445)
63/130 | part-00062.parquet | shape: (200000, 445)
64/130 | part-00063.parquet | shape: (200000, 445)
65/130 | part-00064.parquet | shape: (200000, 445)
66/130 | part-00065.parquet | shape: (200000, 445)
67/130 | part-00066.parquet | shape: (200000, 445)
68/130 | part-00067.parquet | shape: (200000, 445)
69/130 | part-00068.parquet | shape: (200000, 445)
70/130 | part-00069.parquet | shape: (200000, 445)
71/130 | part-00070.parquet | shape: (200000, 445)
72/130 | part-00071.parquet | s

In [ ]:
import os
import polars as pl

bad_files = []
good_files = []

for f in files:
    try:
        schema = pl.read_parquet_schema(f)
        good_files.append(f)
    except Exception as e:
        bad_files.append((f, str(e)))

print("Good files:", len(good_files))
print("Bad files:", len(bad_files))

for f, error in bad_files:
    print("\nBad file:", os.path.basename(f))
    print("Error:", error)

Good files: 129
Bad files: 1

Bad file: part-00051.parquet
Error: parquet: File out of specification: A parquet file must contain a header and footer with at least 12 bytes


In [ ]:
import os
import polars as pl

input_file = "/content/comiset_dataset/content/comiset_parquet_v10/part-00051.parquet"
output_file = "/content/comiset_parquet_v11/part-00051.parquet"

df = pl.read_parquet(input_file)

cols_to_drop = [
    c for c in DROP_COLUMNS
    if c in df.columns
]

df = df.drop(cols_to_drop)

df.write_parquet(
    output_file,
    compression="zstd"
)

print("Recreated:", os.path.basename(output_file))
print("Shape:", df.shape)

Recreated: part-00051.parquet
Shape: (200000, 445)


In [ ]:
import os
import polars as pl

files = sorted([
    os.path.join(OUTPUT_DIR, f)
    for f in os.listdir(OUTPUT_DIR)
    if f.endswith(".parquet")
])

bad_files = []

for f in files:
    try:
        schema = pl.read_parquet_schema(f)

        if len(schema) != 445:
            bad_files.append(
                (os.path.basename(f), f"Columns = {len(schema)}")
            )

        remaining = set(DROP_COLUMNS) & set(schema.keys())

        if remaining:
            bad_files.append(
                (os.path.basename(f), f"Drop columns still present: {remaining}")
            )

    except Exception as e:
        bad_files.append(
            (os.path.basename(f), str(e))
        )

print("Total files:", len(files))
print("Bad files:", len(bad_files))

if bad_files:
    print("\nProblems found:")
    for item in bad_files:
        print(item)
else:
    print("\nEverything is OK!")

Total files: 130
Bad files: 0

Everything is OK!


In [ ]:
import os

OUTPUT_DIR = "/content/comiset_parquet_v11"

total_size = 0
file_count = 0

for root, dirs, files in os.walk(OUTPUT_DIR):
    for file in files:
        path = os.path.join(root, file)

        if os.path.isfile(path):
            total_size += os.path.getsize(path)
            file_count += 1

print("Files:", file_count)
print("Size (GB):", round(total_size / (1024**3), 2))

Files: 130
Size (GB): 0.9


In [ ]:
!zip -r /content/comiset_parquet_v11.zip /content/comiset_parquet_v11


  adding: content/comiset_parquet_v11/ (stored 0%)
  adding: content/comiset_parquet_v11/part-00009.parquet (deflated 4%)
  adding: content/comiset_parquet_v11/part-00108.parquet (deflated 3%)
  adding: content/comiset_parquet_v11/part-00024.parquet (deflated 2%)
  adding: content/comiset_parquet_v11/part-00072.parquet (deflated 3%)
  adding: content/comiset_parquet_v11/part-00069.parquet (deflated 4%)
  adding: content/comiset_parquet_v11/part-00028.parquet (deflated 3%)
  adding: content/comiset_parquet_v11/part-00085.parquet (deflated 4%)
  adding: content/comiset_parquet_v11/part-00027.parquet (deflated 3%)
  adding: content/comiset_parquet_v11/part-00004.parquet (deflated 3%)
  adding: content/comiset_parquet_v11/part-00084.parquet (deflated 4%)
  adding: content/comiset_parquet_v11/part-00030.parquet (deflated 3%)
  adding: content/comiset_parquet_v11/part-00081.parquet (deflated 3%)
  adding: content/comiset_parquet_v11/part-00043.parquet (deflated 2%)
  adding: content/comiset_

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil

shutil.copy(
    "/content/comiset_parquet_v11.zip",
    "/content/drive/MyDrive/comiset_parquet_v11.zip"
)

print("Upload to Google Drive completed!")

FileNotFoundError: [Errno 2] No such file or directory: '/content/comiset_parquet_v11.zip'

In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/comiset_parquet_v11.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction completed!")
print("Folder exists:", os.path.exists("/content/comiset_parquet_v11"))

Extraction completed!
Folder exists: False


In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/comiset_parquet_v11.zip"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    names = zip_ref.namelist()

print("First 10 items:")
for name in names[:10]:
    print(name)

First 10 items:
content/comiset_parquet_v11/
content/comiset_parquet_v11/part-00009.parquet
content/comiset_parquet_v11/part-00108.parquet
content/comiset_parquet_v11/part-00024.parquet
content/comiset_parquet_v11/part-00072.parquet
content/comiset_parquet_v11/part-00069.parquet
content/comiset_parquet_v11/part-00028.parquet
content/comiset_parquet_v11/part-00085.parquet
content/comiset_parquet_v11/part-00027.parquet
content/comiset_parquet_v11/part-00004.parquet


In [ ]:
import glob

INPUT_DIR = "/content/content/comiset_parquet_v11"

FILES = sorted(glob.glob(f"{INPUT_DIR}/*.parquet"))

print("Number of files:", len(FILES))
print("First file:", FILES[0])
print("Last file:", FILES[-1])

Number of files: 130
First file: /content/content/comiset_parquet_v11/part-00000.parquet
Last file: /content/content/comiset_parquet_v11/part-00129.parquet


#feature engineering

In [ ]:
FEATURE_ENGINEERING_COLUMNS = [
    "CallTrace",
    "z_elastic_ecs",
    "Details",
    "TargetFilename",
    "TargetObject",
    "event_original_message",
    "object_name",

    "CommandLine",
    "HiveName",
    "Message",
    "process_path",
    "target_process_path",
    "ActionName",
    "ImagePath","ImageLoaded","Operation",
    "Binary",
    "process_parent_path","src_ip_addr","QueryResults",
    "parentComandLine","etl_pipeline","Keywords","Param1",
]

In [ ]:
import polars as pl
schema = pl.read_parquet_schema(FILES[0])

present = [c for c in FEATURE_ENGINEERING_COLUMNS if c in schema]
missing = [c for c in FEATURE_ENGINEERING_COLUMNS if c not in schema]

print("Present:", len(present))
print("Missing:", len(missing))

print("\nPresent columns:")
print(present)

print("\nMissing columns:")
print(missing)

Present: 20
Missing: 4

Present columns:
['CallTrace', 'z_elastic_ecs', 'Details', 'TargetFilename', 'TargetObject', 'event_original_message', 'object_name', 'CommandLine', 'HiveName', 'Message', 'process_path', 'target_process_path', 'ActionName', 'ImagePath', 'ImageLoaded', 'Operation', 'Binary', 'QueryResults', 'process_parent_path', 'src_ip_addr']

Missing columns:
['parentComandLine', 'etl_pipeline', 'Keywords', 'Param1']


In [ ]:
import glob
import polars as pl

INPUT_DIR = "/content/comiset_dataset/comiset_parquet_v12"
FILES = sorted(glob.glob(f"{INPUT_DIR}/*.parquet"))

print("Number of files:", len(FILES))

Number of files: 130


###"CommandLine" col

In [ ]:
col = "CommandLine"

total_rows = 0
total_non_null = 0

for f in FILES:
    df = pl.read_parquet(f, columns=[col])

    rows = df.height
    non_null = df[col].drop_nulls().len()

    total_rows += rows
    total_non_null += non_null

print("Total rows:", total_rows)
print("Total non-null:", total_non_null)
print("Missing:", total_rows - total_non_null)
print("Non-null percentage:", round(total_non_null / total_rows * 100, 4), "%")

Total rows: 25814017
Total non-null: 20575
Missing: 25793442
Non-null percentage: 0.0797 %


In [ ]:
col = "CommandLine"

commandlines = []

for f in FILES:
    values = (
        pl.read_parquet(f, columns=[col])[col]
        .drop_nulls()
        .to_list()
    )

    commandlines.extend(values)

print("Total non-null values:", len(commandlines))
print("Unique values:", len(set(commandlines)))

print("\nFirst 20 unique CommandLine values:")
for value in list(set(commandlines))[:20]:
    print(value)

Total non-null values: 20575
Unique values: 6682

First 20 unique CommandLine values:
c:\windows\system32\cmd.exe /c echo 52fb053303a > \\.\pipe\55f38d
c:\windows\microsoft.net\framework64\v4.0.30319\mscorsvw.exe -startupevent 90 -interruptevent 0 -ngenprocess 1b0 -pipe 94 -comment "ngen worker process"
c:\windows\microsoft.net\framework\v4.0.30319\mscorsvw.exe -startupevent 2ec -interruptevent 0 -ngenprocess 2e0 -pipe 2b0 -comment "ngen worker process"
c:\windows\microsoft.net\framework64\v4.0.30319\mscorsvw.exe -startupevent 4f0 -interruptevent 0 -ngenprocess 50c -pipe 4b0 -comment "ngen worker process"
c:\windows\system32\bytecodegenerator.exe 2876c419-e8cd-4992-a3c5-f331c227cff3
"c:\windows\microsoft.net\framework\v4.0.30319\ngen.exe" uninstall "c:\windows\assembly\nativeimages_v4.0.30319_32\mmcex\cf16725b4376c34b55c0eb67e06f7715\mmcex.ni.dll" /noroot /legacyservicebehavior
c:\windows\microsoft.net\framework64\v4.0.30319\mscorsvw.exe -startupevent 26c -interruptevent 0 -ngenprocess

  cmd_present

  cmd_length

  cmd_argument_count

  cmd_has_pipe

  cmd_has_encoded_flag
  
  cmd_has_temp_path    

###"ImagePath" col

In [ ]:
col = "ImagePath"

total_rows = 0
total_non_null = 0
unique_values = set()

for f in FILES:
    df = pl.read_parquet(f, columns=[col])

    values = df[col].drop_nulls().to_list()

    total_rows += df.height
    total_non_null += len(values)
    unique_values.update(values)

print("Total rows:", total_rows)
print("Total non-null:", total_non_null)
print("Missing:", total_rows - total_non_null)
print("Non-null percentage:", round(total_non_null / total_rows * 100, 4), "%")
print("Unique values:", len(unique_values))

print("\nFirst 20 unique ImagePath values:")
for value in list(unique_values)[:20]:
    print(value)

Total rows: 25814017
Total non-null: 242
Missing: 25813775
Non-null percentage: 0.0009 %
Unique values: 53

First 20 unique ImagePath values:
"c:\program files\microsoft update health tools\uhssvc.exe"
%systemroot%\system32\vm3dservice.exe
system32\drivers\vmci.sys
c:\programdata\microsoft\windows defender\definition updates\{cf9c746a-9440-47fa-9e90-cc31902adf06}\mpksldrv.sys
\systemroot\system32\drivers\vmmouse.sys
\systemroot\system32\drivers\bthport.sys
c:\programdata\microsoft\windows defender\definition updates\{94326c0b-0be3-46dd-ac15-1dfef3e1075a}\mpksldrv.sys
c:\programdata\microsoft\windows defender\definition updates\{a10a7ba9-0b24-46d8-a6ca-ad4795fd28de}\mpksldrv.sys
\systemroot\system32\drivers\vmrawdsk.sys
c:\programdata\microsoft\windows defender\definition updates\{da4ff22c-4bb0-440c-b659-7816146db28e}\mpksldrv.sys
\systemroot\system32\drivers\e1i65x64.sys
c:\programdata\microsoft\windows defender\definition updates\{590c59b0-0430-4047-95b5-39c05396a125}\mpksldrv.sys
"c:

In [ ]:
col = "ImagePath"

unique_values = set()

for f in FILES:
    values = (
        pl.read_parquet(f, columns=[col])[col]
        .drop_nulls()
        .to_list()
    )
    unique_values.update(values)

print("Unique ImagePath values:", len(unique_values))

for value in sorted(unique_values):
    print(value)

Unique ImagePath values: 53
"c:\program files (x86)\microsoft winlogbeat\winlogbeat\winlogbeat.exe" --environment=windows_service -c "c:\program files (x86)\microsoft winlogbeat\winlogbeat\winlogbeat.yml" --path.home "c:\program files (x86)\microsoft winlogbeat\winlogbeat" --path.data "c:\programdata\winlogbeat" --path.logs "c:\programdata\winlogbeat\logs" -e logging.files.redirect_stderr=true
"c:\program files (x86)\mozilla maintenance service\maintenanceservice.exe"
"c:\program files\microsoft update health tools\uhssvc.exe"
"c:\program files\vmware\vmware tools\vmtoolsd.exe"
"c:\program files\vmware\vmware tools\vmware vgauth\vgauthservice.exe"
%systemroot%\system32\vm3dservice.exe
\\127.0.0.1\admin$\d07fc71.exe
\\desktop-4pvps6e\admin$\841920a.exe
\systemroot\sysmondrv.sys
\systemroot\system32\drivers\bthenum.sys
\systemroot\system32\drivers\bthpan.sys
\systemroot\system32\drivers\bthport.sys
\systemroot\system32\drivers\bthusb.sys
\systemroot\system32\drivers\e1i65x64.sys
\systemr

imagepath_present

imagepath_length

imagepath_is_driver

imagepath_is_network_path

### "ImageLoaded" col

In [ ]:
col = "ImageLoaded"

unique_values = set()
total_rows = 0
total_non_null = 0

for f in FILES:
    df = pl.read_parquet(f, columns=[col])

    values = df[col].drop_nulls().to_list()

    total_rows += df.height
    total_non_null += len(values)
    unique_values.update(values)

print("Total rows:", total_rows)
print("Total non-null:", total_non_null)
print("Missing:", total_rows - total_non_null)
print("Non-null percentage:", round(total_non_null / total_rows * 100, 4), "%")
print("Unique values:", len(unique_values))

print("\nUnique ImageLoaded values:")
for value in sorted(unique_values, key=str):
    print(value)

Total rows: 25814017
Total non-null: 34458
Missing: 25779559
Non-null percentage: 0.1335 %
Unique values: 1230

Unique ImageLoaded values:
\\127.0.0.1\admin$\d07fc71.exe
\\127.0.0.1\admin$\ef9a30b.exe
\\desktop-4pvps6e\admin$\841920a.exe
\\desktop-4pvps6e\admin$\b4fabde.exe
c:\program files (x86)\microsoft\temp\eu4d3d.tmp\microsoftedgeupdate.exe
c:\program files (x86)\microsoft\temp\eu4d3d.tmp\msedgeupdate.dll
c:\program files (x86)\microsoft\temp\eu618e.tmp\microsoftedgeupdate.exe
c:\program files (x86)\microsoft\temp\eu618e.tmp\msedgeupdate.dll
c:\program files (x86)\microsoft\temp\eu74c9.tmp\microsoftedgeupdate.exe
c:\program files (x86)\microsoft\temp\eu74c9.tmp\msedgeupdate.dll
c:\program files (x86)\microsoft\temp\eu78f1.tmp\microsoftedgeupdate.exe
c:\program files (x86)\microsoft\temp\eu78f1.tmp\msedgeupdate.dll
c:\program files (x86)\microsoft\temp\euad9b.tmp\microsoftedgeupdate.exe
c:\program files (x86)\microsoft\temp\euad9b.tmp\msedgeupdate.dll
c:\program files (x86)\microso

In [ ]:
from collections import Counter
import os

col = "ImageLoaded"

extensions = Counter()

for f in FILES:
    values = (
        pl.read_parquet(f, columns=[col])[col]
        .drop_nulls()
        .to_list()
    )

    for value in values:
        filename = str(value).split("\\")[-1]
        ext = os.path.splitext(filename)[1].lower()

        if ext:
            extensions[ext] += 1
        else:
            extensions["<no extension>"] += 1

print("Extensions:")
for ext, count in extensions.most_common():
    print(f"{ext}: {count}")

Extensions:
.dll: 29098
.sys: 4382
.exe: 957
.so: 12
.tmp: 8
.ocx: 1


imageloaded_present

imageloaded_length

imageloaded_is_dll

imageloaded_is_driver

imageloaded_is_executable

imageloaded_is_network_path

### "Operation" col

In [ ]:
col = "Operation"

unique_values = set()
total_rows = 0
total_non_null = 0

for f in FILES:
    df = pl.read_parquet(f, columns=[col])

    values = df[col].drop_nulls().to_list()

    total_rows += df.height
    total_non_null += len(values)
    unique_values.update(values)

print("Total rows:", total_rows)
print("Total non-null:", total_non_null)
print("Missing:", total_rows - total_non_null)
print("Non-null percentage:", round(total_non_null / total_rows * 100, 4), "%")
print("Unique values:", len(unique_values))

print("\nUnique Operation values:")
for value in sorted(unique_values, key=str):
    print(value)

Total rows: 25814017
Total non-null: 4528
Missing: 25809489
Non-null percentage: 0.0175 %
Unique values: 230

Unique Operation values:
Service started
Service stopped
Start IWbemServices::CreateInstanceEnum - root\Microsoft\Windows\Storage : CIM_Error
Start IWbemServices::CreateInstanceEnum - root\Microsoft\Windows\Storage\MS_409 : CIM_Error
Start IWbemServices::CreateInstanceEnum - root\cimv2 : Win32_BIOS
Start IWbemServices::CreateInstanceEnum - root\cimv2 : Win32_ComputerSystem
Start IWbemServices::CreateInstanceEnum - root\cimv2 : Win32_OperatingSystem
Start IWbemServices::CreateInstanceEnum - root\cimv2 : Win32_PageFileUsage
Start IWbemServices::CreateInstanceEnum - root\wmi : MS_SM_AdapterInformationQuery
Start IWbemServices::CreateInstanceEnum - root\wmi : MSiSCSI_PortalInfoClass
Start IWbemServices::DeleteInstance - Root\Rsop\User\S_1_5_21_1311114680_1922563338_3976463898_1000 : RSOP_ExtensionStatus.extensionGuid="{0E28E245-9368-4853-AD84-6DA3BA35BB75}"
Start IWbemServices::Del

operation_present

operation_is_service

operation_has_wmi

###"Binary" col

In [ ]:

import polars as pl

col = "Binary"

total_rows = 0
total_non_null = 0
files_with_values = 0
examples = []

for f in FILES:
    schema = pl.read_parquet_schema(f)

    if col not in schema:
        continue

    df = pl.read_parquet(f, columns=[col])

    total_rows += df.height
    non_null = df[col].drop_nulls()
    count = len(non_null)

    total_non_null += count

    if count > 0:
        files_with_values += 1


        for value in non_null.head(3).to_list():
            text_value = str(value)
            examples.append(text_value[:200])

print("Column:", col)
print("Total rows:", total_rows)
print("Non-null values:", total_non_null)
print("Missing values:", total_rows - total_non_null)

if total_rows > 0:
    print("Non-null percentage:", round(total_non_null / total_rows * 100, 6), "%")

print("Files containing values:", files_with_values)
print("\nSample values:")
for i, value in enumerate(examples[:20], start=1):
    print(f"{i}. {value!r}")


Column: Binary
Total rows: 25814017
Non-null values: 1214
Missing values: 25812803
Non-null percentage: 0.004703 %
Files containing values: 3

Sample values:
1. '620000000A0034000000010003000100010001000100000003000100010024940CA871480C5CE6B7DFE04C59FC1467B063A905417F6E5BD6A0A10D0001000101000201000301000401000500000601000700000801000901000A01000B00000C01003600'
2. '640000000A0036000000010003000100010002000100000003000100010024940CA871480C5CE6B7DFE04C59FC1467B001D163A905417F6E5BD6A0A10D0001000101000201000301000401000500000601000700000801000901000A01000B00000C0100'
3. '2D20436F64653A2020434F525356434330303030303737302D2043616C6C3A2020434F525356434330303030303735322D205049443A202030303030333533322D205449443A202030303030333534342D20434D443A2020433A5C57696E646F77735C73'
4. '8B0100C0'
5. 'E6070B00030010000B000E002A00E6000000000000000000'
6. '00000000020028000000000003000840030008400000000000000000000000000000000000000000'
7. '000004000200300000000000200004600000000000000000000000000000000000

no feature engineering

###"CallTrace" col

In [ ]:

import polars as pl

col = "CallTrace"

total_rows = 0
total_non_null = 0
files_with_values = 0
examples = []

for f in FILES:
    schema = pl.read_parquet_schema(f)

    if col not in schema:
        continue

    df = pl.read_parquet(f, columns=[col])

    total_rows += df.height
    non_null = df[col].drop_nulls()
    count = len(non_null)

    total_non_null += count

    if count > 0:
        files_with_values += 1

        for value in non_null.head(3).to_list():
            text_value = str(value)
            examples.append(text_value[:200])

print("Column:", col)
print("Total rows:", total_rows)
print("Non-null values:", total_non_null)
print("Missing values:", total_rows - total_non_null)

if total_rows > 0:
    print("Non-null percentage:", round(total_non_null / total_rows * 100, 6), "%")

print("Files containing values:", files_with_values)
print("\nSample values:")
for i, value in enumerate(examples[:20], start=1):
    print(f"{i}. {value!r}")


Column: CallTrace
Total rows: 25814017
Non-null values: 5018009
Missing values: 20796008
Non-null percentage: 19.439086 %
Files containing values: 128

Sample values:
1. 'C:\\Windows\\SYSTEM32\\ntdll.dll+93a8a|C:\\Windows\\system32\\KERNELBASE.dll+1f6fd|C:\\ProgramData\\Microsoft\\Windows Defender\\Definition Updates\\{6545CA98-A9D3-4C66-A251-1E91220C2976}\\mpengine.dll+3eacf1|C:\\'
2. 'C:\\Windows\\SYSTEM32\\ntdll.dll+93a8a|C:\\Windows\\system32\\KERNELBASE.dll+1f6fd|C:\\Program Files\\Windows Defender\\mprtp.dll+38f0|C:\\Program Files\\Windows Defender\\mprtp.dll+3a62|C:\\Program Files\\Windows '
3. 'C:\\Windows\\SYSTEM32\\ntdll.dll+93a8a|C:\\Windows\\system32\\KERNELBASE.dll+1f6fd|C:\\Program Files\\Windows Defender\\mprtp.dll+38f0|C:\\Program Files\\Windows Defender\\mprtp.dll+3a62|C:\\Program Files\\Windows '
4. 'C:\\Windows\\SYSTEM32\\ntdll.dll+93a8a|C:\\Windows\\system32\\KERNELBASE.dll+1f6fd|C:\\Program Files\\Windows Defender\\mprtp.dll+38f0|C:\\Program Files\\Windows Defender

calltrace_present

calltrace_length

calltrace_module_count

calltrace_has_defender

###"z_elastic_ecs" col

In [ ]:

import polars as pl

col = "z_elastic_ecs"

total_rows = 0
total_non_null = 0
files_with_values = 0
examples = []

for f in FILES:
    schema = pl.read_parquet_schema(f)

    if col not in schema:
        continue

    df = pl.read_parquet(f, columns=[col])

    total_rows += df.height
    non_null = df[col].drop_nulls()
    count = len(non_null)

    total_non_null += count

    if count > 0:
        files_with_values += 1

        for value in non_null.head(3).to_list():
            text_value = str(value)
            examples.append(text_value[:200])

print("Column:", col)
print("Total rows:", total_rows)
print("Non-null values:", total_non_null)
print("Missing values:", total_rows - total_non_null)

if total_rows > 0:
    print("Non-null percentage:", round(total_non_null / total_rows * 100, 6), "%")

print("Files containing values:", files_with_values)
print("\nSample values:")
for i, value in enumerate(examples[:20], start=1):
    print(f"{i}. {value!r}")


Column: z_elastic_ecs
Total rows: 25814017
Non-null values: 25814017
Missing values: 0
Non-null percentage: 100.0 %
Files containing values: 130

Sample values:
1. '{"agent":{},"event":{"created":"2022-11-16T08:28:14.318Z","code":"5858","provider":"Microsoft-Windows-WMI-Activity","kind":"event","action":"None"},"host":{},"ecs":{"version":"8.0.0"},"log":{},"winlog'
2. '{"agent":{},"event":{"created":"2022-11-16T08:28:14.319Z","code":"5858","provider":"Microsoft-Windows-WMI-Activity","action":"None","kind":"event"},"host":{},"ecs":{"version":"8.0.0"},"log":{},"winlog'
3. '{"agent":{},"event":{"created":"2022-11-16T08:28:14.318Z","code":"5858","provider":"Microsoft-Windows-WMI-Activity","action":"None","kind":"event"},"host":{},"ecs":{"version":"8.0.0"},"log":{},"winlog'
4. '{"agent":{},"ecs":{"version":"8.0.0"},"host":{},"log":{},"event":{"action":"Audit Policy Change","outcome":"success","provider":"Microsoft-Windows-Security-Auditing","code":"4907","kind":"event","crea'
5. '{"agent":{}

ecs_json_length

#"Details" col

In [ ]:
import polars as pl

col = "Details"

total_rows = 0
total_non_null = 0
files_with_values = 0
examples = []

for f in FILES:
    schema = pl.read_parquet_schema(f)

    if col not in schema:
        continue

    df_temp = pl.read_parquet(f, columns=[col])
    total_rows += df_temp.height

    non_null = df_temp[col].drop_nulls()
    count = len(non_null)
    total_non_null += count

    if count > 0:
        files_with_values += 1

        for value in non_null.head(3).to_list():
            examples.append(str(value)[:250])

print("Column:", col)
print("Total rows:", total_rows)
print("Non-null values:", total_non_null)
print("Missing values:", total_rows - total_non_null)

if total_rows:
    print(
        "Non-null percentage:",
        round(total_non_null / total_rows * 100, 6),
        "%"
    )

print("Files containing values:", files_with_values)
print("\nSample values:")

for i, value in enumerate(examples[:20], start=1):
    print(f"{i}. {value!r}")

Column: Details
Total rows: 25814017
Non-null values: 673161
Missing values: 25140856
Non-null percentage: 2.607734 %
Files containing values: 129

Sample values:
1. 'The service will auto stop if no requests received for some period of time.'
2. 'The service will auto stop if no requests received for some period of time.'
3. 'The service has stopped.'
4. 'DWORD (0x00000003)'
5. 'DWORD (0x00000264)'
6. 'DWORD (0x00000001)'
7. 'DWORD (0x00000000)'
8. '{14E74C62-DC97-43B0-8F2F-581496A65D60}'
9. 'Traditional Chinese Phone Converter'
10. 'DWORD (0x000008b8)'
11. 'DWORD (0x000008b8)'
12. 'DWORD (0x000008b8)'
13. 'DWORD (0x6374fdea)'
14. 'DWORD (0x6374fb47)'
15. 'DWORD (0x00000000)'
16. 'Binary Data'
17. 'Binary Data'
18. 'Binary Data'
19. 'DWORD (0x6374d9cd)'
20. 'DWORD (0x00000000)'


In [ ]:
import polars as pl

total_rows = 0
present = 0
dword_count = 0
binary_data_count = 0
hex_count = 0
lengths = []

for f in FILES:
    schema = pl.read_parquet_schema(f)

    if "Details" not in schema:
        continue

    data = pl.read_parquet(f, columns=["Details"])
    details = pl.col("Details").cast(pl.String)

    result = data.select([
        pl.len().alias("rows"),

        details.is_not_null()
            .sum()
            .alias("present"),

        details.str.contains(
            r"(?i)^\s*DWORD\s*\(\s*0x[0-9a-f]+\s*\)\s*$"
        ).fill_null(False)
         .sum()
         .alias("dword"),

        details.str.to_lowercase()
            .str.contains("binary data", literal=True)
            .fill_null(False)
            .sum()
            .alias("binary_data"),

        details.str.contains(
            r"(?i)0x[0-9a-f]+"
        ).fill_null(False)
         .sum()
         .alias("hex"),

        details.str.len_chars()
            .filter(details.is_not_null())
            .median()
            .alias("median_length")
    ])

    total_rows += result["rows"][0]
    present += result["present"][0]
    dword_count += result["dword"][0]
    binary_data_count += result["binary_data"][0]
    hex_count += result["hex"][0]

    median_length = result["median_length"][0]
    if median_length is not None:
        lengths.append(median_length)

print("Total rows:", total_rows)
print("Non-null Details:", present)
print("DWORD values:", dword_count)
print("Binary Data values:", binary_data_count)
print("Hex-containing values:", hex_count)
print("Median length per file:", lengths[:10])

Total rows: 25814017
Non-null Details: 673161
DWORD values: 322506
Binary Data values: 115896
Hex-containing values: 427495
Median length per file: [75.0, 18.0, 15.0, 18.0, 18.0, 18.0, 18.0, 18.0, 18.0, 18.0]


In [ ]:
import polars as pl

dword_examples = []
hex_non_dword_examples = []
binary_examples = []

for f in FILES:
    schema = pl.read_parquet_schema(f)

    if "Details" not in schema:
        continue

    data = pl.read_parquet(f, columns=["Details"])

    dword = pl.col("Details").cast(pl.String).str.contains(
        r"(?i)^\s*DWORD\s*\(\s*0x[0-9a-f]+\s*\)\s*$"
    ).fill_null(False)

    has_hex = pl.col("Details").cast(pl.String).str.contains(
        r"(?i)0x[0-9a-f]+"
    ).fill_null(False)

    binary = pl.col("Details").cast(pl.String).str.to_lowercase().eq(
        "binary data"
    ).fill_null(False)

    dword_examples += (
        data.filter(dword)["Details"].head(3).to_list()
    )

    hex_non_dword_examples += (
        data.filter(has_hex & ~dword)["Details"].head(3).to_list()
    )

    binary_examples += (
        data.filter(binary)["Details"].head(3).to_list()
    )

    if (
        len(dword_examples) >= 10
        and len(hex_non_dword_examples) >= 10
        and len(binary_examples) >= 10
    ):
        break

print("DWORD examples:", dword_examples[:10])
print("Hex but not DWORD:", hex_non_dword_examples[:10])
print("Binary Data examples:", binary_examples[:10])

DWORD examples: ['DWORD (0x00000003)', 'DWORD (0x00000264)', 'DWORD (0x00000001)', 'DWORD (0x00000000)', 'DWORD (0x00000000)', 'DWORD (0x00000000)', 'DWORD (0x000008b8)', 'DWORD (0x000008b8)', 'DWORD (0x000008b8)', 'DWORD (0x6374fdea)']
Hex but not DWORD: ['AQAAANCMnd8BFdERjHoAwE/Cl+sBAAAAFfOBera8hk+L5Jsod0HHdgAAAAACAAAAAAAQZgAAAAEAACAAAADHO16eao21GAGpS5ZgVSPsNy6LdlbGiQqMSbi2IV9puAAAAAAOgAAAAAIAACAAAAD1WvPdxtu7vKQIPsQeYkS6Sxu+uoTXB3U7RpwjJANN+wASAABeb5tGrYmQM4E0oJSfI5k2nWXSAPCv6eQUxryCU4sFvwdsQikz+SqLNErm5liby/hFze9jVWiWJlpk0oTwaJbP9NQVxTsvix5AIW64nr9E8YBYsTHXc4W9FOFedhQ/RlZq6XrxFUTQ1PYiqUd4M1TVAdq+gmX7a9DdeJx5E6JDeLClXsUeA6tq4YIQTTAwl1z6D7HxeKa/0SZLUhT6uttAMDkGZbumr+LgUfhUxf02g53D3l5RJj/wuKukAN16pAP+L9+KByNRCCgBdbfXdG6iNcj2weVCR6H0DvzFkMNFr3/R9UW2s8Wb7tne2vHv9vVRR34yxgEzzQCWqAcY7byu1/4TZtZ0ZT7eu5oLw2GQlrqwy+3ICceP2pkNV5geAxW3SXzO61yiow0UGamnVn1jZpf6bQmk/SLgflTE0VtN13CNo3epJ0friwcA3i9ykS6iqkz/o2mnt4wdIUskX7+1b12a4LYeSCu3/q8tf7L7bHObcAyV04/DX3sI2b9j+zTQvEJVZe1rrNiE6xTqX57IP9k+6TQEi2dVLy

details_present

details_length

details_is_dword

details_is_qword

details_is_binary_data

#"TargetFilename" col

In [ ]:
import polars as pl

col = "TargetFilename"

total_rows = 0
total_non_null = 0
files_with_values = 0
examples = []

for f in FILES:
    schema = pl.read_parquet_schema(f)

    if col not in schema:
        continue

    df_temp = pl.read_parquet(f, columns=[col])
    total_rows += df_temp.height

    non_null = df_temp[col].drop_nulls()
    count = len(non_null)
    total_non_null += count

    if count > 0:
        files_with_values += 1

        for value in non_null.head(3).to_list():
            examples.append(str(value)[:250])

print("Column:", col)
print("Total rows:", total_rows)
print("Non-null values:", total_non_null)
print("Missing values:", total_rows - total_non_null)

if total_rows:
    print(
        "Non-null percentage:",
        round(total_non_null / total_rows * 100, 6),
        "%"
    )

print("Files containing values:", files_with_values)
print("\nSample values:")

for i, value in enumerate(examples[:20], start=1):
    print(f"{i}. {value!r}")

Column: TargetFilename
Total rows: 25814017
Non-null values: 951693
Missing values: 24862324
Non-null percentage: 3.68673 %
Files containing values: 128

Sample values:
1. 'c:\\windows\\prefetch\\cmd.exe-0bd30981.pf'
2. 'c:\\windows\\prefetch\\mobsync.exe-b307e1cc.pf'
3. 'c:\\windows\\prefetch\\wsapphelper.exe-301f9286.pf'
4. 'c:\\windows\\system32\\wdi\\{9f41811a-0429-42aa-81b7-cfd4d968411f}\\{e7aa9db1-95df-4bc8-b572-49720f673c8d}\\snapshot.etl'
5. 'c:\\windows\\system32\\wdi\\{9f41811a-0429-42aa-81b7-cfd4d968411f}\\{e7aa9db1-95df-4bc8-b572-49720f673c8d}'
6. 'c:\\windows\\system32\\config\\systemprofile\\appdata\\local\\microsoft\\clr_v4.0\\usagelogs\\tzsync.exe.log'
7. 'c:\\windows\\prefetch\\mmc.exe-a3710740.pf'
8. 'c:\\windows\\prefetch\\dllhost.exe-d9dcd0f3.pf'
9. 'c:\\windows\\prefetch\\mmc.exe-a3710740.pf'
10. 'c:\\windows\\prefetch\\wmiprvse.exe-e8b8dd29.pf'
11. 'c:\\windows\\prefetch\\wmiprvse.exe-e8b8dd29.pf'
12. 'c:\\windows\\prefetch\\wmiprvse.exe-e8b8dd29.pf'
13. 'c:\\wind

In [ ]:

import polars as pl

patterns = {
    "prefetch": r"(?i)\\prefetch\\",
    "temp": r"(?i)\\windows\\temp\\",
    "system32": r"(?i)\\system32\\",
    "windows_logs": r"(?i)\\windows\\logs\\",
    "executable": r"(?i)\.exe$",
    "log_or_etl": r"(?i)\.(log|etl)$",
    "registry_related": r"(?i)\\(config\\|regback\\)",
}

counts = {name: 0 for name in patterns}
total = 0
non_null = 0
length_sum = 0
length_max = 0

for f in FILES:
    schema = pl.read_parquet_schema(f)
    if "TargetFilename" not in schema:
        continue

    col = pl.read_parquet(
        f, columns=["TargetFilename"]
    )["TargetFilename"].cast(pl.String)

    total += len(col)
    non_null += col.is_not_null().sum()

    length = col.str.len_chars()
    length_sum += length.sum()
    length_max = max(
        length_max, length.max() or 0
    )

    for name, pattern in patterns.items():
        counts[name] += (
            col.str.contains(pattern)
            .fill_null(False)
            .sum()
        )

print("Total rows:", total)
print("Non-null:", non_null)
print("Missing:", total - non_null)
print("Mean path length (including missing as zero):",
      round(length_sum / total, 3))

print("\nPattern counts:")
for name, count in counts.items():
    print(f"{name}: {count:,} ({count / total * 100:.4f}%)")

print("Maximum path length:", length_max)


Total rows: 25814017
Non-null: 951693
Missing: 24862324
Mean path length (including missing as zero): 4.394

Pattern counts:
prefetch: 18,321 (0.0710%)
temp: 44,360 (0.1718%)
system32: 42,414 (0.1643%)
windows_logs: 3,265 (0.0126%)
executable: 4,240 (0.0164%)
log_or_etl: 7,953 (0.0308%)
registry_related: 23,261 (0.0901%)
Maximum path length: 283


In [ ]:

import polars as pl

patterns = {
    "executable": r"(?i)\.exe$",
    "registry_related": r"(?i)\\(config\\|regback\\)",
}

total_non_null = 0
length_sum = 0
length_max = 0
length_min = None
examples = {name: [] for name in patterns}

for f in FILES:
    schema = pl.read_parquet_schema(f)
    if "TargetFilename" not in schema:
        continue

    col = pl.read_parquet(
        f, columns=["TargetFilename"]
    )["TargetFilename"].cast(pl.String)

    valid = col.drop_nulls()
    total_non_null += len(valid)

    lengths = valid.str.len_chars()
    if len(lengths):
        length_sum += lengths.sum()
        length_max = max(length_max, lengths.max())
        current_min = lengths.min()
        length_min = (
            current_min if length_min is None
            else min(length_min, current_min)
        )

    for name, pattern in patterns.items():
        matched = valid.filter(
            valid.str.contains(pattern).fill_null(False)
        )
        for value in matched.head(
            max(0, 10 - len(examples[name]))
        ).to_list():
            examples[name].append(value)

print("Non-null values:", total_non_null)
print("Mean path length:", round(length_sum / total_non_null, 2))
print("Minimum path length:", length_min)
print("Maximum path length:", length_max)

for name, values in examples.items():
    print(f"\n{name} examples:")
    for value in values:
        print(repr(value))


Non-null values: 951693
Mean path length: 119.19
Minimum path length: 15
Maximum path length: 283

executable examples:
'c:\\users\\louis\\appdata\\local\\microsoft\\onedrive\\update\\onedrivesetup.exe'
'c:\\windows\\softwaredistribution\\download\\install\\am_delta_patch_1.379.432.0.exe'
'c:\\windows\\temp\\dbb58b09-f7a4-415c-bf67-6fc8751f4e61\\dismhost.exe'
'c:\\windows\\temp\\868ccb74-cff1-4bf3-ae8d-1fbaade41848\\dismhost.exe'
'c:\\windows\\temp\\dbb58b09-f7a4-415c-bf67-6fc8751f4e61\\dismhost.exe'
'c:\\windows\\temp\\396e03f2-c9d7-45b7-96e8-d29af10ce9c6\\dismhost.exe'
'c:\\windows\\temp\\15590048-eeaf-4f69-a409-223981942456\\dismhost.exe'
'c:\\windows\\winsxs\\amd64_adobe-flash-for-windows_31bf3856ad364e35_10.0.10240.16384_none_33a055b5043cde6d\\flashutil_activex.exe'
'c:\\windows\\system32\\driverstore\\filerepository\\bth.inf_amd64_6e5d1d4c2e1fa2ec\\fsquirt.exe'
'c:\\program files (x86)\\mozilla maintenance service\\maintenanceservice_tmp.exe'

registry_related examples:
'c:\\wind

targetfilename_present

targetfilename_length

target_is_prefetch

target_is_temp

target_is_system32

target_is_windows_logs

target_is_executable

target_is_log_file

target_is_registry_backup

target_is_system_config

#"TargetObject" col

In [ ]:
import polars as pl

col = "TargetObject"

total_rows = 0
total_non_null = 0
files_with_values = 0
examples = []

for f in FILES:
    schema = pl.read_parquet_schema(f)

    if col not in schema:
        continue

    df_temp = pl.read_parquet(f, columns=[col])
    total_rows += df_temp.height

    non_null = df_temp[col].drop_nulls()
    count = len(non_null)
    total_non_null += count

    if count > 0:
        files_with_values += 1

        for value in non_null.head(3).to_list():
            examples.append(str(value)[:250])

print("Column:", col)
print("Total rows:", total_rows)
print("Non-null values:", total_non_null)
print("Missing values:", total_rows - total_non_null)

if total_rows:
    print(
        "Non-null percentage:",
        round(total_non_null / total_rows * 100, 6),
        "%"
    )

print("Files containing values:", files_with_values)
print("\nSample values:")

for i, value in enumerate(examples[:20], start=1):
    print(f"{i}. {value!r}")

Column: TargetObject
Total rows: 25814017
Non-null values: 18149032
Missing values: 7664985
Non-null percentage: 70.306888 %
Files containing values: 128

Sample values:
1. 'HKLM\\SOFTWARE\\MICROSOFT\\EnterpriseCertificates\\Disallowed\\CRLs'
2. 'HKU\\.DEFAULT\\Software\\Microsoft\\Windows\\CurrentVersion\\WinTrust\\Trust Providers\\Software Publishing'
3. 'HKU\\.DEFAULT\\Software\\Policies\\Microsoft\\SystemCertificates\\Disallowed\\CRLs'
4. 'HKLM\\System\\CurrentControlSet\\Services\\Tcpip\\Parameters'
5. 'HKLM\\System\\CurrentControlSet\\Services\\Tcpip\\Parameters'
6. 'HKLM\\System\\CurrentControlSet\\Services\\Tcpip\\Parameters'
7. 'HKU\\S-1-5-21-1311114680-1922563338-3976463898-1000_CLASSES\\Local Settings\\Software\\Microsoft\\Windows\\CurrentVersion\\AppContainer\\Storage\\microsoft.windows.cortana_cw5n1h2txyewy\\Internet Explorer\\DOMStorage\\bing.com'
8. 'HKU\\S-1-5-21-1311114680-1922563338-3976463898-1000_CLASSES\\Local Settings\\Software\\Microsoft\\Windows\\CurrentVersion\

In [ ]:

import polars as pl

patterns = {
    "hklm": r"(?i)^HKLM\\",
    "hku": r"(?i)^HKU\\",
    "tcpip": r"(?i)\\Tcpip\\",
    "certificate": r"(?i)certificate|\\CRLs\\",
    "services": r"(?i)\\Services\\",
    "software": r"(?i)\\Software\\",
    "appcontainer": r"(?i)\\AppContainer\\",
}

total_rows = 0
total_non_null = 0
length_sum = 0
length_max = 0
counts = {name: 0 for name in patterns}
examples = {name: [] for name in patterns}

for f in FILES:
    schema = pl.read_parquet_schema(f)
    if "TargetObject" not in schema:
        continue

    col = pl.read_parquet(
        f, columns=["TargetObject"]
    )["TargetObject"].cast(pl.String)

    total_rows += len(col)
    valid = col.drop_nulls()
    total_non_null += len(valid)

    lengths = valid.str.len_chars()
    if len(lengths):
        length_sum += lengths.sum()
        length_max = max(length_max, lengths.max())

    for name, pattern in patterns.items():
        matched = valid.filter(
            valid.str.contains(pattern).fill_null(False)
        )
        counts[name] += len(matched)

        remaining = 5 - len(examples[name])
        if remaining > 0:
            examples[name].extend(
                matched.head(remaining).to_list()
            )

print("Total rows:", total_rows)
print("Non-null:", total_non_null)
print("Missing:", total_rows - total_non_null)
print("Mean non-null length:",
      round(length_sum / total_non_null, 2))
print("Maximum length:", length_max)

print("\nPattern counts:")
for name, count in counts.items():
    print(f"{name}: {count:,} ({count / total_rows * 100:.3f}%)")

for name, values in examples.items():
    print(f"\n{name} examples:")
    for value in values:
        print(repr(value))


Total rows: 25814017
Non-null: 18149032
Missing: 7664985
Mean non-null length: 61.05
Maximum length: 336

Pattern counts:
hklm: 15,580,512 (60.357%)
hku: 2,391,656 (9.265%)
tcpip: 12,791,336 (49.552%)
certificate: 3,840,681 (14.878%)
services: 12,954,865 (50.185%)
software: 4,709,208 (18.243%)
appcontainer: 20,068 (0.078%)

hklm examples:
'HKLM\\SOFTWARE\\MICROSOFT\\EnterpriseCertificates\\Disallowed\\CRLs'
'HKLM\\SOFTWARE\\MICROSOFT\\EnterpriseCertificates\\Disallowed\\Certificates'
'HKLM\\SOFTWARE\\Policies\\Microsoft\\SystemCertificates\\Disallowed'
'HKLM\\SOFTWARE\\MICROSOFT\\EnterpriseCertificates\\Disallowed\\CTLs'
'HKLM\\SOFTWARE\\Policies\\Microsoft\\SystemCertificates\\Disallowed\\CTLs'

hku examples:
'HKU\\.DEFAULT\\Software\\Microsoft\\Windows\\CurrentVersion\\WinTrust\\Trust Providers\\Software Publishing'
'HKU\\.DEFAULT\\Software\\Policies\\Microsoft\\SystemCertificates\\Disallowed\\CRLs'
'HKU\\.DEFAULT\\Software\\Microsoft\\Windows\\CurrentVersion\\WinTrust\\Trust Provide

targetobject_present

targetobject_length

targetobject_is_hklm

targetobject_is_hku

targetobject_is_tcpip

targetobject_is_certificate

targetobject_is_services

targetobject_is_software

targetobject_is_appcontainer

#"event_original_message" col

In [ ]:
import polars as pl

col = "event_original_message"

examples = []
files_checked = 0

for f in FILES:
    schema = pl.read_parquet_schema(f)

    if col not in schema:
        continue

    try:
        sample = (
            pl.scan_parquet(f)
            .select(col)
            .head(100)
            .collect(engine="streaming")
        )

        files_checked += 1

        remaining = 20 - len(examples)
        if remaining > 0:
            values = sample[col].drop_nulls().head(
                remaining
            ).to_list()

            examples.extend(
                str(v)[:200] for v in values
            )

        del sample

        if len(examples) >= 20:
            break

    except Exception as e:
        print("File failed:", f)
        print("Error:", repr(e))
        break

print("Files sampled:", files_checked)
print("\nSample values:")

for i, value in enumerate(examples, start=1):
    print(f"{i}. {value!r}")


Files sampled: 1

Sample values:
1. 'Id = {00000000-0000-0000-0000-000000000000}; ClientMachine = DESKTOP-4PVPS6E; User = NT AUTHORITY\\SYSTEM; ClientProcessId = 4264; Component = Unknown; Operation = Start IWbemServices::ExecQuery - root'
2. 'Id = {00000000-0000-0000-0000-000000000000}; ClientMachine = DESKTOP-4PVPS6E; User = NT AUTHORITY\\SYSTEM; ClientProcessId = 4264; Component = Unknown; Operation = Start IWbemServices::ExecQuery - root'
3. 'Id = {00000000-0000-0000-0000-000000000000}; ClientMachine = DESKTOP-4PVPS6E; User = NT AUTHORITY\\SYSTEM; ClientProcessId = 2248; Component = Unknown; Operation = Start IWbemServices::ExecQuery - root'
4. 'Id = {00000000-0000-0000-0000-000000000000}; ClientMachine = DESKTOP-4PVPS6E; User = NT AUTHORITY\\SYSTEM; ClientProcessId = 4264; Component = Unknown; Operation = Start IWbemServices::ExecQuery - ROOT'
5. 'Id = {00000000-0000-0000-0000-000000000000}; ClientMachine = DESKTOP-4PVPS6E; User = NT AUTHORITY\\SYSTEM; ClientProcessId = 4264; Comp

event_message_present

event_message_length

event_has_error

event_has_failure

event_mentions_defender

event_mentions_wmi

event_mentions_process_id

event_mentions_ip

#"object_name" col

In [ ]:
import polars as pl

col = "object_name"

total_rows = 0
total_non_null = 0
files_with_values = 0
examples = []

for f in FILES:
    schema = pl.read_parquet_schema(f)

    if col not in schema:
        continue

    df_temp = pl.read_parquet(f, columns=[col])
    total_rows += df_temp.height

    non_null = df_temp[col].drop_nulls()
    count = len(non_null)
    total_non_null += count

    if count > 0:
        files_with_values += 1

        for value in non_null.head(10).to_list():
            examples.append(str(value)[:250])

print("Column:", col)
print("Total rows:", total_rows)
print("Non-null values:", total_non_null)
print("Missing values:", total_rows - total_non_null)

if total_rows:
    print(
        "Non-null percentage:",
        round(total_non_null / total_rows * 100, 6),
        "%"
    )

print("Files containing values:", files_with_values)
print("\nSample values:")

for i, value in enumerate(examples[:20], start=1):
    print(f"{i}. {value!r}")

Column: object_name
Total rows: 25814017
Non-null values: 20504
Missing values: 25793513
Non-null percentage: 0.07943 %
Files containing values: 2

Sample values:
1. 'c:\\windows\\temp\\winre\\extractedfromwim'
2. 'c:\\windows\\winsxs\\filemaps\\_0000000000000000.cdf-ms'
3. 'c:\\windows\\winsxs\\filemaps\\$$_microsoft.net_3296b36dbe4c7fa3.cdf-ms'
4. 'c:\\windows\\winsxs\\filemaps\\$$_microsoft.net_framework_83386eac0379231b.cdf-ms'
5. 'c:\\windows\\winsxs\\filemaps\\$$_microsoft.net_framework_v4.0.30319_wpf_en-us_dc5fd125966afabc.cdf-ms'
6. 'c:\\windows\\winsxs\\filemaps\\$$_microsoft.net_assembly_gac_msil_uiautomationprovider_v4.0_4.0.0.0_31bf3856ad364e35_6bb637099f04ee2c.cdf-ms'
7. 'c:\\windows\\winsxs\\filemaps\\$$_microsoft.net_framework64_v4.0.30319_wpf_fonts_0428e0346460ac4c.cdf-ms'
8. 'c:\\windows\\winsxs\\filemaps\\$$_microsoft.net_assembly_gac_msil_system.windows.controls.ribbon_v4.0_4.0.0.0_b77a5c561934e089_f0c023acb7bafe74.cdf-ms'
9. 'c:\\windows\\winsxs\\filemaps\\$$_micros

object_name_present

object_name_length

object_is_temp

object_is_system32

object_is_executable

#"HiveName" col

In [ ]:
import polars as pl

col = "HiveName"

total_rows = 0
total_non_null = 0
files_with_values = 0
examples = []

for f in FILES:
    schema = pl.read_parquet_schema(f)

    if col not in schema:
        continue

    df_temp = pl.read_parquet(f, columns=[col])
    total_rows += df_temp.height

    non_null = df_temp[col].drop_nulls()
    count = len(non_null)
    total_non_null += count

    if count > 0:
        files_with_values += 1

        for value in non_null.head(16).to_list():
            examples.append(str(value)[:250])

print("Column:", col)
print("Total rows:", total_rows)
print("Non-null values:", total_non_null)
print("Missing values:", total_rows - total_non_null)

if total_rows:
    print(
        "Non-null percentage:",
        round(total_non_null / total_rows * 100, 6),
        "%"
    )

print("Files containing values:", files_with_values)
print("\nSample values:")

for i, value in enumerate(examples[:20], start=1):
    print(f"{i}. {value!r}")

Column: HiveName
Total rows: 25814017
Non-null values: 1708
Missing values: 25812309
Non-null percentage: 0.006617 %
Files containing values: 2

Sample values:
1. '\\??\\c:\\users\\louis\\appdata\\local\\packages\\microsoft.windowscommunicationsapps_8wekyb3d8bbwe\\settings\\settings.dat'
2. '\\??\\c:\\windows\\system32\\smi\\store\\machine\\schema.dat'
3. '\\??\\c:\\windows\\system32\\smi\\store\\machine\\schema.dat'
4. '\\??\\c:\\windows\\system32\\smi\\store\\machine\\schema.dat'
5. '\\??\\c:\\windows\\system32\\config\\elam'
6. '\\??\\c:\\$windows.~bt\\sources\\rollback\\efi\\microsoft\\boot\\bcd'
7. '\\??\\c:\\programdata\\microsoft\\windows\\apprepository\\packages\\f46d4000-fd22-4db4-ac8e-4e1ddde828fe_10.0.19041.423_neutral_neutral_cw5n1h2txyewy\\activationstore.dat'
8. '\\??\\c:\\programdata\\microsoft\\windows\\apprepository\\packages\\microsoft.windows.capturepicker_10.0.19041.423_neutral__cw5n1h2txyewy\\activationstore.dat'
9. '\\??\\c:\\programdata\\microsoft\\windows\\appre

hivename_present

hivename_length

hivename_is_system32

hivename_is_config

hivename_is_bcd

hivename_is_dat

#"Message" col

In [ ]:
import polars as pl

col = "Message"

total_rows = 0
total_non_null = 0
files_with_values = 0
examples = []

for f in FILES:
    schema = pl.read_parquet_schema(f)

    if col not in schema:
        continue

    df_temp = pl.read_parquet(f, columns=[col])
    total_rows += df_temp.height

    non_null = df_temp[col].drop_nulls()
    count = len(non_null)
    total_non_null += count

    if count > 0:
        files_with_values += 1

        for value in non_null.head(16).to_list():
            examples.append(str(value)[:250])

print("Column:", col)
print("Total rows:", total_rows)
print("Non-null values:", total_non_null)
print("Missing values:", total_rows - total_non_null)

if total_rows:
    print(
        "Non-null percentage:",
        round(total_non_null / total_rows * 100, 6),
        "%"
    )

print("Files containing values:", files_with_values)
print("\nSample values:")

for i, value in enumerate(examples[:20], start=1):
    print(f"{i}. {value!r}")

Column: Message
Total rows: 25814017
Non-null values: 71153
Missing values: 25742864
Non-null percentage: 0.275637 %
Files containing values: 3

Sample values:
1. 'SearchForAllUpdates request: interactive = false'
2. 'PackageFamilyName: Microsoft.Appconnector_8wekyb3d8bbwe, type: 2, ContentId: 1A7994D6-5342-8581-71FB-A2BD1C895D93'
3. 'PackageFamilyName: Microsoft.BingNews_8wekyb3d8bbwe, type: 2, ContentId: 1F63B8C3-2D48-9497-0A0A-2CBD462EDE76'
4. 'PackageFamilyName: Microsoft.BingWeather_8wekyb3d8bbwe, type: 2, ContentId: E6658C19-4221-2EBE-763A-F0493FBA2BB0'
5. 'PackageFamilyName: Microsoft.MicrosoftSolitaireCollection_8wekyb3d8bbwe, type: 2, ContentId: 00D57B0F-01FA-B79F-08D6-878ED20C4C9B'
6. 'PackageFamilyName: Microsoft.SkypeApp_kzf8qxf38zg5c, type: 2, ContentId: E6D3B497-80AF-7F14-F9E6-9606EE369FC3'
7. 'PackageFamilyName: Microsoft.WindowsSoundRecorder_8wekyb3d8bbwe, type: 2, ContentId: 0C8CF327-9D17-CCDE-18AF-DFF4F20070E5'
8. 'PackageFamilyName: Microsoft.XboxApp_8wekyb3d8bbwe, t

message_present

message_length

message_has_error

message_has_failure

message_has_memory_error

message_has_hex_code

message_mentions_package

#"process_path" col

In [ ]:
import polars as pl

col = "process_path"

total_rows = 0
total_non_null = 0
files_with_values = 0
examples = []

for f in FILES:
    schema = pl.read_parquet_schema(f)

    if col not in schema:
        continue

    df_temp = pl.read_parquet(f, columns=[col])
    total_rows += df_temp.height

    non_null = df_temp[col].drop_nulls()
    count = len(non_null)
    total_non_null += count

    if count > 0:
        files_with_values += 1

        for value in non_null.head(3).to_list():
            examples.append(str(value)[:250])

print("Column:", col)
print("Total rows:", total_rows)
print("Non-null values:", total_non_null)
print("Missing values:", total_rows - total_non_null)

if total_rows:
    print(
        "Non-null percentage:",
        round(total_non_null / total_rows * 100, 6),
        "%"
    )

print("Files containing values:", files_with_values)
print("\nSample values:")

for i, value in enumerate(examples[:20], start=1):
    print(f"{i}. {value!r}")

Column: process_path
Total rows: 25814017
Non-null values: 25604427
Missing values: 209590
Non-null percentage: 99.188077 %
Files containing values: 130

Sample values:
1. 'c:\\windows\\system32\\svchost.exe'
2. 'c:\\windows\\system32\\svchost.exe'
3. '\\device\\harddiskvolume3\\windows\\system32\\spoolsv.exe'
4. 'c:\\windows\\system32\\poqexec.exe'
5. 'c:\\windows\\system32\\poqexec.exe'
6. 'c:\\windows\\system32\\poqexec.exe'
7. 'c:\\program files\\windows defender\\msmpeng.exe'
8. 'c:\\program files\\windows defender\\msmpeng.exe'
9. 'c:\\program files\\windows defender\\msmpeng.exe'
10. 'c:\\program files\\windows defender\\msmpeng.exe'
11. 'c:\\program files\\windows defender\\msmpeng.exe'
12. 'c:\\program files\\windows defender\\msmpeng.exe'
13. 'c:\\program files (x86)\\remote mouse\\remotemouse.exe'
14. 'c:\\program files (x86)\\remote mouse\\remotemouse.exe'
15. 'c:\\program files (x86)\\remote mouse\\remotemouse.exe'
16. 'c:\\windows\\system32\\srtasks.exe'
17. 'c:\\program 

process_path_present

process_path_length

process_is_system32

process_is_windows_defender

process_is_program_files

process_is_temp

process_is_executable

#"target_process_path" col

In [ ]:
import polars as pl

col = "target_process_path"

total_rows = 0
total_non_null = 0
files_with_values = 0
examples = []

for f in FILES:
    schema = pl.read_parquet_schema(f)

    if col not in schema:
        continue

    df_temp = pl.read_parquet(f, columns=[col])
    total_rows += df_temp.height

    non_null = df_temp[col].drop_nulls()
    count = len(non_null)
    total_non_null += count

    if count > 0:
        files_with_values += 1

        for value in non_null.head(2).to_list():
            examples.append(str(value)[:250])

print("Column:", col)
print("Total rows:", total_rows)
print("Non-null values:", total_non_null)
print("Missing values:", total_rows - total_non_null)

if total_rows:
    print(
        "Non-null percentage:",
        round(total_non_null / total_rows * 100, 6),
        "%"
    )

print("Files containing values:", files_with_values)
print("\nSample values:")

for i, value in enumerate(examples[:20], start=1):
    print(f"{i}. {value!r}")

Column: target_process_path
Total rows: 25814017
Non-null values: 5024458
Missing values: 20789559
Non-null percentage: 19.464069 %
Files containing values: 129

Sample values:
1. 'registry'
2. 'registry'
3. 'registry'
4. 'registry'
5. 'c:\\program files (x86)\\remote mouse\\remotemouse.exe'
6. 'c:\\program files (x86)\\remote mouse\\remotemouse.exe'
7. 'c:\\program files (x86)\\remote mouse\\remotemouse.exe'
8. 'c:\\program files (x86)\\remote mouse\\remotemouse.exe'
9. 'c:\\program files (x86)\\remote mouse\\remotemouse.exe'
10. 'c:\\program files (x86)\\remote mouse\\remotemouse.exe'
11. 'c:\\windows\\system32\\conhost.exe'
12. 'c:\\windows\\system32\\vssvc.exe'
13. 'c:\\program files (x86)\\remote mouse\\remotemouse.exe'
14. 'c:\\windows\\system32\\searchindexer.exe'
15. 'c:\\program files (x86)\\remote mouse\\remotemouse.exe'
16. 'c:\\users\\louis\\desktop\\tools\\beacon.exe'
17. 'c:\\program files (x86)\\remote mouse\\remotemouse.exe'
18. 'c:\\program files (x86)\\remote mouse\\r

target_process_path_present

target_process_path_length

target_process_is_registry

target_process_is_system32

target_process_is_windows_defender

target_process_is_temp

target_process_is_executable

#"ActionName" col

In [ ]:
import polars as pl

col = "ActionName"

total_rows = 0
total_non_null = 0
files_with_values = 0
examples = []

for f in FILES:
    schema = pl.read_parquet_schema(f)

    if col not in schema:
        continue

    df_temp = pl.read_parquet(f, columns=[col])
    total_rows += df_temp.height

    non_null = df_temp[col].drop_nulls()
    count = len(non_null)
    total_non_null += count

    if count > 0:
        files_with_values += 1

        for value in non_null.head(20).to_list():
            examples.append(str(value)[:250])

print("Column:", col)
print("Total rows:", total_rows)
print("Non-null values:", total_non_null)
print("Missing values:", total_rows - total_non_null)

if total_rows:
    print(
        "Non-null percentage:",
        round(total_non_null / total_rows * 100, 6),
        "%"
    )

print("Files containing values:", files_with_values)
print("\nSample values:")

for i, value in enumerate(examples[:20], start=1):
    print(f"{i}. {value!r}")

Column: ActionName
Total rows: 25814017
Non-null values: 6689
Missing values: 25807328
Non-null percentage: 0.025912 %
Files containing values: 1

Sample values:
1. '%windir%\\system32\\wermgr.exe'
2. 'network state change task handler'
3. 'network state change task handler'
4. 'network state change task handler'
5. '%systemroot%\\system32\\musnotification.exe'
6. 'network state change task handler'
7. '%systemroot%\\system32\\sihclient.exe'
8. 'c:\\program files (x86)\\mozilla firefox\\firefox.exe'
9. 'group policy background processing'
10. '%localappdata%\\microsoft\\onedrive\\onedrivestandaloneupdater.exe'
11. '%systemroot%\\system32\\sihclient.exe'
12. '%windir%\\system32\\wermgr.exe'
13. 'c:\\windows\\system32\\msfeedssync.exe'
14. 'certificate services client task handler'
15. 'language components installer'
16. 'work folder logon trigger class'
17. 'network state change task handler'
18. 'c:\\windows\\system32\\musnotification.exe'
19. '%localappdata%\\microsoft\\onedrive\\oned

actionname_present

actionname_length

actionname_is_executable_path

actionname_is_network_task

actionname_is_group_policy

#"QueryResults" col

In [ ]:
import polars as pl

col = "QueryResults"

total_rows = 0
total_non_null = 0
files_with_values = 0
examples = []

for f in FILES:
    schema = pl.read_parquet_schema(f)

    if col not in schema:
        continue

    df_temp = pl.read_parquet(f, columns=[col])
    total_rows += df_temp.height

    non_null = df_temp[col].drop_nulls()
    count = len(non_null)
    total_non_null += count

    if count > 0:
        files_with_values += 1

        for value in non_null.head(3).to_list():
            examples.append(str(value)[:250])

print("Column:", col)
print("Total rows:", total_rows)
print("Non-null values:", total_non_null)
print("Missing values:", total_rows - total_non_null)

if total_rows:
    print(
        "Non-null percentage:",
        round(total_non_null / total_rows * 100, 6),
        "%"
    )

print("Files containing values:", files_with_values)
print("\nSample values:")

for i, value in enumerate(examples[:20], start=1):
    print(f"{i}. {value!r}")

Column: QueryResults
Total rows: 25814017
Non-null values: 8739
Missing values: 25805278
Non-null percentage: 0.033854 %
Files containing values: 128

Sample values:
1. 'type:  5 fe2.update.msft.com.trafficmanager.net;::ffff:52.238.248.2;::ffff:52.254.114.65;'
2. 'type:  5 platform-bing-com.dual-a-0034.a-msedge.net;type:  5 dual-a-0034.a-msedge.net;::ffff:204.79.197.237;::ffff:13.107.21.237;'
3. 'type:  5 assets.msn.com.edgekey.net;type:  5 e28578.d.akamaiedge.net;::ffff:2.18.63.225;::ffff:2.18.63.213;'
4. 'type:  5 v10-win.vortex.data.trafficmanager.net;type:  5 onedscolprduks05.uksouth.cloudapp.azure.com;::ffff:51.132.193.105;'
5. 'type:  5 v10-win.vortex.data.trafficmanager.net;type:  5 onedscolprdweu03.westeurope.cloudapp.azure.com;::ffff:13.69.109.131;'
6. 'type:  5 v10-win.vortex.data.trafficmanager.net;type:  5 onedscolprdcus00.centralus.cloudapp.azure.com;::ffff:13.89.178.26;'
7. 'type:  5 wu-fg-shim.trafficmanager.net;type:  5 download.windowsupdate.com.edgesuite.net;type:  5 

queryresults_present

queryresults_length

queryresults_domain_count

queryresults_ip_count

queryresults_has_ipv6_mapped

queryresults_has_trafficmanager

queryresults_has_windows_update

#"process_parent_path" col

In [ ]:
import polars as pl

col = "process_parent_path"

total_rows = 0
total_non_null = 0
files_with_values = 0
examples = []

for f in FILES:
    schema = pl.read_parquet_schema(f)

    if col not in schema:
        continue

    df_temp = pl.read_parquet(f, columns=[col])
    total_rows += df_temp.height

    non_null = df_temp[col].drop_nulls()
    count = len(non_null)
    total_non_null += count

    if count > 0:
        files_with_values += 1

        for value in non_null.head(3).to_list():
            examples.append(str(value)[:250])

print("Column:", col)
print("Total rows:", total_rows)
print("Non-null values:", total_non_null)
print("Missing values:", total_rows - total_non_null)

if total_rows:
    print(
        "Non-null percentage:",
        round(total_non_null / total_rows * 100, 6),
        "%"
    )

print("Files containing values:", files_with_values)
print("\nSample values:")

for i, value in enumerate(examples[:20], start=1):
    print(f"{i}. {value!r}")

Column: process_parent_path
Total rows: 25814017
Non-null values: 21203
Missing values: 25792814
Non-null percentage: 0.082138 %
Files containing values: 129

Sample values:
1. 'c:\\windows\\system32\\smss.exe'
2. 'c:\\windows\\system32\\smss.exe'
3. 'c:\\windows\\system32\\smss.exe'
4. 'c:\\windows\\system32\\smss.exe'
5. 'c:\\windows\\system32\\wininit.exe'
6. 'c:\\windows\\system32\\smss.exe'
7. 'c:\\windows\\system32\\svchost.exe'
8. 'c:\\windows\\system32\\services.exe'
9. 'c:\\windows\\microsoft.net\\framework64\\v4.0.30319\\ngentask.exe'
10. 'c:\\windows\\system32\\svchost.exe'
11. 'c:\\windows\\explorer.exe'
12. 'c:\\program files\\rempl\\sedlauncher.exe'
13. 'c:\\windows\\system32\\svchost.exe'
14. 'c:\\windows\\system32\\svchost.exe'
15. 'c:\\windows\\system32\\svchost.exe'
16. 'c:\\windows\\system32\\svchost.exe'
17. 'c:\\windows\\system32\\mcbuilder.exe'
18. 'c:\\windows\\system32\\svchost.exe'
19. 'c:\\windows\\system32\\svchost.exe'
20. 'c:\\windows\\system32\\svchost.exe

parent_process_present

parent_process_path_length

parent_is_system32

parent_is_windows_explorer

parent_is_svchost

parent_is_temp

#"src_ip_addr" col

In [ ]:
import polars as pl

col = "src_ip_addr"

total_rows = 0
total_non_null = 0
files_with_values = 0
examples = []

for f in FILES:
    schema = pl.read_parquet_schema(f)

    if col not in schema:
        continue

    df_temp = pl.read_parquet(f, columns=[col])
    total_rows += df_temp.height

    non_null = df_temp[col].drop_nulls()
    count = len(non_null)
    total_non_null += count

    if count > 0:
        files_with_values += 1

        for value in non_null.head(2).to_list():
            examples.append(str(value)[:250])

print("Column:", col)
print("Total rows:", total_rows)
print("Non-null values:", total_non_null)
print("Missing values:", total_rows - total_non_null)

if total_rows:
    print(
        "Non-null percentage:",
        round(total_non_null / total_rows * 100, 6),
        "%"
    )

print("Files containing values:", files_with_values)
print("\nSample values:")

for i, value in enumerate(examples[:20], start=1):
    print(f"{i}. {value!r}")

Column: src_ip_addr
Total rows: 25814017
Non-null values: 1187749
Missing values: 24626268
Non-null percentage: 4.601178 %
Files containing values: 129

Sample values:
1. '10.10.234.128'
2. '10.10.234.128'
3. '10.10.234.1'
4. '::1'
5. '10.10.117.130'
6. '10.10.117.130'
7. '10.10.234.130'
8. '10.10.234.130'
9. '10.10.117.130'
10. '10.10.117.130'
11. '10.10.117.130'
12. '10.10.234.120'
13. '10.10.117.130'
14. '10.10.117.130'
15. '192.168.137.131'
16. '192.168.137.131'
17. '192.168.137.131'
18. '192.168.137.131'
19. '192.168.137.131'
20. '192.168.137.131'


src_ip_present

src_ip_is_private

src_ip_is_loopback

src_ip_is_ipv4

src_ip_is_ipv6

In [ ]:

import os
import glob
import re
import gc
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# =====================================================
# 1. Paths and settings
# =====================================================

INPUT_DIR = "/content/comiset_dataset/comiset_parquet_v12"
OUTPUT_DIR = "/content/comiset_dataset/comiset_parquet_v12_FE"

BATCH_SIZE = 20_000

files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.parquet")))

if len(files) != 130:
    raise ValueError(f"Expected 130 files, found {len(files)}")

# Avoid accidentally overwriting previous results
if os.path.exists(OUTPUT_DIR) and os.listdir(OUTPUT_DIR):
    raise FileExistsError(
        f"Output folder is not empty: {OUTPUT_DIR}\n"
        "Use a new output folder or inspect the existing results first."
    )

os.makedirs(OUTPUT_DIR, exist_ok=True)

FE_COLUMNS = [
    "CommandLine", "ImagePath", "ImageLoaded", "Operation", "Binary",
    "CallTrace", "z_elastic_ecs", "Details", "TargetFilename",
    "TargetObject", "event_original_message", "object_name", "HiveName",
    "Message", "process_path", "target_process_path", "ActionName",
    "QueryResults", "process_parent_path", "src_ip_addr"
]

# Confirm that the input schema contains all selected columns
input_schema = pq.read_schema(files[0])
missing = [c for c in FE_COLUMNS if c not in input_schema.names]

if missing:
    raise ValueError(f"Missing FE columns: {missing}")

KEEP_COLUMNS = [c for c in input_schema.names if c not in FE_COLUMNS]

# =====================================================
# 2. Feature engineering
# =====================================================

def generate_features(df):
    features = {}

    def s(name):
        return df[name].astype("string").fillna("").str.lower()

    def present(name, prefix):
        x = s(name)
        features[f"{prefix}_present"] = (
            x.str.strip().ne("").astype("int8")
        )

    def length(name, feature_name):
        features[feature_name] = (
            s(name).str.len().fillna(0).astype("int32")
        )

    def flag(name, feature_name, pattern, regex=True):
        features[feature_name] = (
            s(name).str.contains(
                pattern, regex=regex, na=False
            ).astype("int8")
        )

    # 1. process_path
    x = s("process_path")
    features["process_path_present"] = x.str.strip().ne("").astype("int8")
    flag("process_path", "process_is_system32", r"\\system32\\")
    flag("process_path", "process_is_program_files",
         r"\\program files(?: \(x86\))?\\")
    flag("process_path", "process_is_temp",
         r"\\temp\\|\\appdata\\local\\temp\\")
    flag("process_path", "process_is_windows_defender",
         r"\\windows defender\\|\\msmpeng\.exe")
    features["process_path_length"] = x.str.len().fillna(0).astype("int32")
    features["process_path_depth"] = x.str.count(r"\\").fillna(0).astype("int16")
    features["process_extension_is_exe"] = x.str.endswith(".exe").astype("int8")

    # 2. target_process_path
    x = s("target_process_path")
    features["target_process_present"] = x.str.strip().ne("").astype("int8")
    features["target_is_registry"] = x.str.strip().eq("registry").astype("int8")
    flag("target_process_path", "target_is_system32", r"\\system32\\")
    flag("target_process_path", "target_is_temp",
         r"\\temp\\|\\appdata\\local\\temp\\")
    flag("target_process_path", "target_is_program_files",
         r"\\program files(?: \(x86\))?\\")
    features["target_process_path_length"] = x.str.len().fillna(0).astype("int32")
    features["target_process_extension_is_exe"] = x.str.endswith(".exe").astype("int8")

    # 3. process_parent_path
    x = s("process_parent_path")
    features["parent_process_present"] = x.str.strip().ne("").astype("int8")
    flag("process_parent_path", "parent_is_system32", r"\\system32\\")
    flag("process_parent_path", "parent_is_temp",
         r"\\temp\\|\\appdata\\local\\temp\\")
    flag("process_parent_path", "parent_is_program_files",
         r"\\program files(?: \(x86\))?\\")
    features["parent_process_extension_is_exe"] = x.str.endswith(".exe").astype("int8")

    # 4. ImagePath
    x = s("ImagePath")
    features["imagepath_present"] = x.str.strip().ne("").astype("int8")
    features["imagepath_is_driver"] = x.str.endswith(".sys").astype("int8")
    features["imagepath_is_exe"] = x.str.endswith(".exe").astype("int8")
    flag("ImagePath", "imagepath_is_system32", r"\\system32\\")
    flag("ImagePath", "imagepath_is_defender", r"defender|msmpeng")

    # 5. object_name
    x = s("object_name")
    features["object_name_present"] = x.str.strip().ne("").astype("int8")
    flag("object_name", "object_is_temp", r"\\temp\\|\\appdata\\local\\temp\\")
    flag("object_name", "object_is_winsxs", r"\\winsxs\\")
    flag("object_name", "object_is_system32", r"\\system32\\")
    features["object_name_length"] = x.str.len().fillna(0).astype("int32")

    # 6. HiveName
    x = s("HiveName")
    features["hivename_present"] = x.str.strip().ne("").astype("int8")
    flag("HiveName", "hivename_is_system_config", r"system|config")
    flag("HiveName", "hivename_is_apprepository", "apprepository", regex=False)
    flag("HiveName", "hivename_is_user_profile", r"users|user profile|hku")
    features["hivename_length"] = x.str.len().fillna(0).astype("int32")

    # 7. TargetObject
    x = s("TargetObject")
    features["targetobject_present"] = x.str.strip().ne("").astype("int8")
    for term, feature_name in [
        ("hklm", "targetobject_is_hklm"),
        ("hku", "targetobject_is_hku"),
        ("tcpip", "targetobject_has_tcpip"),
        ("certificate", "targetobject_has_certificate"),
        ("services", "targetobject_has_services"),
        ("software", "targetobject_has_software"),
        ("appcontainer", "targetobject_has_appcontainer"),
    ]:
        features[feature_name] = x.str.contains(term, regex=False).astype("int8")
    features["targetobject_length"] = x.str.len().fillna(0).astype("int32")
    features["targetobject_depth"] = x.str.count(r"\\").fillna(0).astype("int16")

    # 8. Operation
    x = s("Operation")
    features["operation_present"] = x.str.strip().ne("").astype("int8")
    flag("Operation", "operation_is_service_start", r"start.*service|service.*start")
    flag("Operation", "operation_is_service_stop", r"stop.*service|service.*stop")
    flag("Operation", "operation_is_wmi", "wmi", regex=False)
    features["operation_length"] = x.str.len().fillna(0).astype("int32")

    # 9. Details
    x = s("Details")
    features["details_present"] = x.str.strip().ne("").astype("int8")
    flag("Details", "details_is_binary_data", "binary data", regex=False)
    flag("Details", "details_is_dword", "dword", regex=False)
    flag("Details", "details_is_qword", "qword", regex=False)
    flag("Details", "details_has_hex_value", r"0x[0-9a-f]+")
    features["details_length"] = x.str.len().fillna(0).astype("int32")

    # 10. ActionName
    x = s("ActionName")
    features["actionname_present"] = x.str.strip().ne("").astype("int8")
    flag("ActionName", "actionname_is_executable_path", r"\.exe(?:$|\s)")
    flag("ActionName", "actionname_is_system32", r"\\system32\\")
    flag("ActionName", "actionname_is_task_handler", "taskhandler", regex=False)
    features["actionname_length"] = x.str.len().fillna(0).astype("int32")

    # 11. CommandLine
    x = s("CommandLine")
    features["cmd_present"] = x.str.strip().ne("").astype("int8")
    features["cmd_length"] = x.str.len().fillna(0).astype("int32")
    features["cmd_argument_count"] = x.str.split().str.len().fillna(0).astype("int16")
    flag("CommandLine", "cmd_has_encoded_flag", r"-encodedcommand|-enc\b")
    flag("CommandLine", "cmd_has_powershell", "powershell", regex=False)
    flag("CommandLine", "cmd_has_temp_path", r"\\temp\\|\\appdata\\local\\temp\\")
    flag("CommandLine", "cmd_has_admin_share", r"\\\\[^\\]+\\(?:admin\$|c\$)")

    # 12. ImageLoaded
    x = s("ImageLoaded")
    features["imageloaded_present"] = x.str.strip().ne("").astype("int8")
    features["imageloaded_is_dll"] = x.str.endswith(".dll").astype("int8")
    features["imageloaded_is_sys"] = x.str.endswith(".sys").astype("int8")
    features["imageloaded_is_exe"] = x.str.endswith(".exe").astype("int8")
    flag("ImageLoaded", "imageloaded_is_system32", r"\\system32\\")
    flag("ImageLoaded", "imageloaded_is_temp", r"\\temp\\|\\appdata\\local\\temp\\")
    features["imageloaded_length"] = x.str.len().fillna(0).astype("int32")

    # 13. TargetFilename
    x = s("TargetFilename")
    features["targetfilename_present"] = x.str.strip().ne("").astype("int8")
    features["targetfilename_length"] = x.str.len().fillna(0).astype("int32")
    flag("TargetFilename", "targetfilename_is_prefetch", r"\\prefetch\\")
    flag("TargetFilename", "targetfilename_is_temp", r"\\temp\\|\\appdata\\local\\temp\\")
    flag("TargetFilename", "targetfilename_is_system32", r"\\system32\\")
    features["targetfilename_is_log_or_etl"] = x.str.endswith((".log", ".etl")).astype("int8")
    features["targetfilename_is_executable"] = x.str.endswith((".exe", ".dll", ".sys")).astype("int8")

    # 14. CallTrace
    x = s("CallTrace")
    features["calltrace_present"] = x.str.strip().ne("").astype("int8")
    features["calltrace_length"] = x.str.len().fillna(0).astype("int32")
    flag("CallTrace", "calltrace_has_ntdll", "ntdll", regex=False)
    flag("CallTrace", "calltrace_has_kernelbase", "kernelbase", regex=False)
    flag("CallTrace", "calltrace_has_defender_module", r"mpengine|wd\w+\.dll|defender")

    # Binary: intentionally no feature; original column will be removed
    # as requested for the 20 selected source columns.

    # 15. event_original_message
    x = s("event_original_message")
    features["original_message_present"] = x.str.strip().ne("").astype("int8")
    features["original_message_length"] = x.str.len().fillna(0).astype("int32")
    features["original_message_line_count"] = (
        x.str.count(r"\n").fillna(0) + x.str.strip().ne("").astype("int8")
    ).astype("int16")

    # 16. Message
    x = s("Message")
    features["message_present"] = x.str.strip().ne("").astype("int8")
    features["message_length"] = x.str.len().fillna(0).astype("int32")

    # 17. QueryResults
    x = s("QueryResults")
    features["queryresults_present"] = x.str.strip().ne("").astype("int8")
    features["queryresults_length"] = x.str.len().fillna(0).astype("int32")
    features["queryresults_ip_count"] = x.str.count(
        r"\b(?:\d{1,3}\.){3}\d{1,3}\b|(?:[0-9a-f]{0,4}:){2,}[0-9a-f:]*"
    ).fillna(0).astype("int16")

    # 18. src_ip_addr
    x = s("src_ip_addr")
    features["src_ip_present"] = x.str.strip().ne("").astype("int8")
    features["src_ip_is_private"] = x.str.match(
        r"^(10\.|192\.168\.|172\.(?:1[6-9]|2\d|3[01])\.)",
        na=False
    ).astype("int8")
    features["src_ip_is_loopback"] = x.isin(["127.0.0.1", "::1"]).astype("int8")
    features["src_ip_is_ipv6"] = x.str.contains(":", regex=False).astype("int8")

    # 19. z_elastic_ecs
    x = s("z_elastic_ecs")
    features["ecs_present"] = x.str.strip().ne("").astype("int8")
    features["ecs_length"] = x.str.len().fillna(0).astype("int32")
    flag("z_elastic_ecs", "ecs_has_event_code", '"code"', regex=False)
    flag("z_elastic_ecs", "ecs_has_provider", '"provider"', regex=False)
    flag("z_elastic_ecs", "ecs_has_outcome", '"outcome"', regex=False)

    return pd.DataFrame(features, index=df.index)


# =====================================================
# 3. Process files and batches
# =====================================================

feature_count = None
total_rows = 0
completed = 0

for file_index, input_path in enumerate(files, start=1):
    filename = os.path.basename(input_path)
    output_path = os.path.join(OUTPUT_DIR, filename)

    print(f"\n[{file_index}/130] Processing {filename}", flush=True)

    parquet_file = pq.ParquetFile(input_path)
    input_rows = parquet_file.metadata.num_rows
    writer = None
    file_rows = 0

    try:
        for batch in parquet_file.iter_batches(
            batch_size=BATCH_SIZE,
            columns=input_schema.names
        ):
            # Read only the selected 20 columns into pandas for FE
            fe_table = batch.select(FE_COLUMNS)
            df = fe_table.to_pandas()

            feature_df = generate_features(df)

            # Remove the 20 source columns from the retained table
            output_table = batch.select(KEEP_COLUMNS)

            # Append generated numeric features to the Arrow table
            for col in feature_df.columns:
                output_table = output_table.append_column(
                    col,
                    pa.array(feature_df[col].to_numpy(), from_pandas=False)
                )

            if writer is None:
                writer = pq.ParquetWriter(
                    output_path,
                    output_table.schema,
                    compression="zstd"
                )
                if feature_count is None:
                    feature_count = len(feature_df.columns)

            writer.write_table(
    pa.Table.from_batches([output_table]),
    row_group_size=BATCH_SIZE
)

            batch_rows = output_table.num_rows
            file_rows += batch_rows
            total_rows += batch_rows

            del batch, fe_table, df, feature_df, output_table
            gc.collect()

        if writer is not None:
            writer.close()
            writer = None

        # Verify the saved file
        output_pf = pq.ParquetFile(output_path)
        output_rows = output_pf.metadata.num_rows
        output_cols = len(output_pf.schema_arrow.names)

        expected_cols = len(KEEP_COLUMNS) + feature_count

        if output_rows != input_rows:
            raise RuntimeError(
                f"Row mismatch in {filename}: input={input_rows}, output={output_rows}"
            )

        if output_cols != expected_cols:
            raise RuntimeError(
                f"Column mismatch in {filename}: expected={expected_cols}, got={output_cols}"
            )

        completed += 1
        print(
            f"Verified: {output_rows:,} rows | {output_cols:,} columns "
            f"| {feature_count} new features",
            flush=True
        )

    except Exception:
        if writer is not None:
            writer.close()
        print(f"ERROR while processing {filename}. Stop and inspect output.", flush=True)
        raise

print("\n======================================")
print("FEATURE ENGINEERING FINISHED")
print("Files completed:", completed, "/", len(files))
print("Total output rows:", f"{total_rows:,}")
print("Generated features per row:", feature_count)
print("Output folder:", OUTPUT_DIR)
print("Original input files were not modified.")
print("======================================")



[1/130] Processing part-00000.parquet
Verified: 200,000 rows | 28,206 columns | 104 new features

[2/130] Processing part-00001.parquet
Verified: 200,000 rows | 28,206 columns | 104 new features

[3/130] Processing part-00002.parquet
Verified: 200,000 rows | 28,206 columns | 104 new features

[4/130] Processing part-00003.parquet
Verified: 200,000 rows | 28,206 columns | 104 new features

[5/130] Processing part-00004.parquet
Verified: 200,000 rows | 28,206 columns | 104 new features

[6/130] Processing part-00005.parquet
Verified: 200,000 rows | 28,206 columns | 104 new features

[7/130] Processing part-00006.parquet
Verified: 200,000 rows | 28,206 columns | 104 new features

[8/130] Processing part-00007.parquet
Verified: 200,000 rows | 28,206 columns | 104 new features

[9/130] Processing part-00008.parquet
Verified: 200,000 rows | 28,206 columns | 104 new features

[10/130] Processing part-00009.parquet
Verified: 200,000 rows | 28,206 columns | 104 new features

[11/130] Processin